## tl;dr

مدل غیرخطی سه‌ماهه، فروش شهریور ۱۴۰۵ را در نقطه مرکزی حدود **۴۳۵٫۵ میلیارد تومان** برآورد می‌کند. دامنه سناریوهای تاریخی **۳۴۷ تا ۵۲۴ میلیارد تومان** است.

## Context & Methods

فروش خالص فاکتور و حواله از `dbo.SalesReviewFast` با کسر برگشتی و تبدیل ریال به تومان محاسبه می‌شود. ماه‌های ۱۴۰۵/۰۲، ۱۴۰۵/۰۳ و ۱۴۰۵/۰۵ مبنای منحنی هستند. تیر به‌دلیل داشتن فقط دو روز کامل در بازه ۱–۵ از برازش اصلی کنار گذاشته شده است.

### Key Assumptions

- روزهایی با تعداد سند کمتر از ۲۰٪ میانه همان ماه، ثبت ناقص/غیرعملیاتی تلقی می‌شوند.
- تعداد روزهای کاری شهریور در پنج بازه به‌ترتیب ۵، ۳، ۴، ۴ و ۹ روز است.
- رفتار افزایش فروش در طول ماه به‌صورت میانه ضرایب سه ماه معتبر اعمال می‌شود.

## Data

اجرای دوباره تحلیل فقط‌خواندنی و بارگذاری خروجی کنترل‌شده.

In [ ]:
import contextlib, io, json, runpy
from pathlib import Path

analysis_path = Path('docs/reports/sales_forecast_140506/analysis.py')
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path(str(analysis_path), run_name='__main__')
result = json.loads(Path('docs/reports/sales_forecast_140506/analysis_result.json').read_text(encoding='utf-8'))
result['source_snapshot'], result['quality']

## Results

ضرایب فروش روزانه هر بازه نسبت به روزهای ۱–۵:

In [ ]:
[{'bucket': row['bucket'], 'multiplier': round(row['median_multiplier'], 3), 'increase_percent': round((row['median_multiplier'] - 1) * 100, 1)} for row in result['uplift_curve']]

In [ ]:
[{'bucket': row['bucket'], 'workdays': row['expected_workdays'], 'forecast_bn_toman': round(row['forecast_bucket_toman'] / 1e9, 1)} for row in result['forecast_parts']], {key: round(value / 1e9, 1) for key, value in result['forecast'].items()}

## Takeaways

- پیش‌بینی خطی ۲۵۲ میلیارد تومانی، اثر افزایش فروش در نیمه دوم ماه را حذف می‌کرد.
- ضریب میانه در بازه ۲۱ تا پایان ماه ۲٫۳۲ برابر ابتدای ماه است.
- برآورد مرکزی ۴۳۵٫۵ میلیارد تومان است، اما به‌دلیل فقط سه روز داده جاری و نوسان تاریخی باید با دامنه ۳۴۷ تا ۵۲۴ میلیارد تومان گزارش شود.